# 05 - Hidden-feature steering and method comparison

Notebook `04` obtained a steering direction by differentiating an external
objective at every active step. Here we ask whether a useful direction is
already present inside the frozen velocity network. We record one hidden layer
on labeled, noised examples, test on separate examples whether class can be
decoded from that representation, and learn a target-versus-rest direction
offline. During sampling we add that direction to the hidden feature rather
than to the sample position, so the downstream layers decide how the velocity
changes and no inference-time gradient is needed.


In [ ]:
import sys
from pathlib import Path

NOTES_DIR = Path.cwd()
if not (NOTES_DIR / "toy_notes.py").exists():
    NOTES_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(NOTES_DIR.resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_notes import *
from notebook_views import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print("device:", device)


In [ ]:
model, losses, _ = load_or_train_model(device=device)
reference_data, reference_labels = sample_labeled_mixture(30000, device=device)
stats = estimate_gaussian_stats(reference_data, reference_labels)
TARGET_CLASS = 2
model.eval()
_ = model.requires_grad_(False)


## 1. Why intervene inside the network?

The velocity MLP first converts the current state `(t, x_t)` into a hidden
feature vector $h_t$. The intervention changes that feature before the final
layers compute velocity:

$$h'_t = h_t + \alpha\,g(t)\,\operatorname{RMS}(h_t)\,d_{0.90}.$$

Here $d_{0.90}$ is a unit target-vs-rest direction learned from a labeled
activation training split and checked on separate held-out activations,
$\alpha$ controls strength, and $g(t)$ turns the edit on only in a chosen time
window. The RMS factor puts the unit direction on the scale of the current
layer.

This does **not** pull the sample toward a point. It changes an internal feature,
which may change the model's velocity; the ODE sampler then accumulates those
velocity changes. Model weights stay frozen.


In [ ]:
plot_activation_pipeline()


## 2. When is class information visible in the hidden layer?

At each time, we record activations from separate balanced training and test
sets with independent noise. A linear probe is fitted only on the training
activations and scored on the untouched test activations. A shuffled-label
probe provides a simple comparison without the true class relationship.
The comparison asks whether class is readable from the hidden state; it does
not yet show that editing the state will control generation.


In [ ]:
def collect_activations(model, *, t_value, n_per_class):
    labels = torch.arange(len(CLASS_NAMES), device=device).repeat_interleave(n_per_class)
    x_data, labels = sample_labeled_mixture(
        len(labels), device=device, labels=labels,
    )
    x_noise = sample_base(len(labels), device=device)
    t = torch.full((len(labels),), t_value, device=device)
    x_t = (1 - t[:, None]) * x_noise + t[:, None] * x_data
    with torch.no_grad():
        features = model.features(t, x_t)
    return features, labels


In [ ]:
REFERENCE_T = 0.90
set_seed(SEED + 500)
train_features, train_labels = collect_activations(
    model, t_value=REFERENCE_T, n_per_class=600,
)
set_seed(SEED + 501)
test_features, test_labels = collect_activations(
    model, t_value=REFERENCE_T, n_per_class=400,
)

probe = fit_linear_probe(train_features, train_labels)
probe_accuracy = (probe.predict(test_features) == test_labels).float().mean()

positive = train_features[train_labels == TARGET_CLASS]
negative = train_features[train_labels != TARGET_CLASS]
mean_difference = positive.mean(0) - negative.mean(0)
centered = train_features - train_features.mean(0)
covariance = centered.T @ centered / (len(centered) - 1)
ridge = 0.1 * torch.eye(covariance.shape[0], device=device)
reference_direction = torch.linalg.solve(covariance + ridge, mean_difference)
reference_direction /= reference_direction.norm()

shuffled_labels = train_labels[torch.randperm(len(train_labels), device=device)]
shuffled_probe = fit_linear_probe(train_features, shuffled_labels)
shuffled_accuracy = (shuffled_probe.predict(test_features) == test_labels).float().mean()
shuffled_direction = discriminant_direction(train_features, shuffled_labels, TARGET_CLASS)

display(pd.Series({
    "held-out probe accuracy": float(probe_accuracy),
    "shuffled-label accuracy": float(shuffled_accuracy),
}, name=f"t={REFERENCE_T}").round(3))


In [ ]:
pca_display = fit_pca_projection(train_features)
projected_test = pca_display.transform(test_features)

fig, ax = plt.subplots(figsize=(6, 5))
plot_labeled_points(
    projected_test, test_labels, ax=ax,
    title=f"Held-out activations at t={REFERENCE_T}", alpha=0.35,
)
ax.legend(frameon=False)
plt.show()


### Reading fixed-direction transfer

We estimate $d_{0.90}$ at late time and reuse it as a fixed intervention vector
over the window $[0.40, 0.90]$. The cosine plot shows that time-specific
directions are only weakly aligned with $d_{0.90}$ near the beginning of that
window. The intervention therefore does not follow the local class direction
at every time. It asks a narrower question: can one late-time direction,
repeated over part of the trajectory, produce a useful endpoint change?


In [ ]:
probe_rows, directions = activation_probe_sweep(
    model,
    target_class=TARGET_CLASS,
    times=[0.10, 0.25, 0.40, 0.55, 0.70, 0.85, 0.90],
    seed=SEED + 700,
    collect=collect_activations,
)
probe_table = pd.DataFrame(probe_rows)
display(probe_table.round(3))
plot_activation_probe_sweep(
    probe_table, directions, reference_direction, reference_t=REFERENCE_T,
)


## 3. Activation intervention

The target direction is covariance-aware: it separates target examples from
the remaining classes in hidden space. During an active time window, the model
adds a scaled unit direction to the selected hidden layer. The scale is relative
to the current feature RMS, so it remains comparable across time.

This is a genuine hidden-feature intervention, not sample-space attraction. It
is still simpler than NA-RFM: the paper uses RFM directions in image-model
activation tensors, while this notebook uses a linear discriminant direction in
one MLP layer.

We collect the direction at `t = 0.90`, where the held-out probe is strongest,
then reuse it over a wider intervention window. Collection time and
intervention time are different choices: the cosine plot above shows that the
late direction is not the local class direction at every step. The paired
comparison below asks whether this fixed edit is nevertheless useful.


In [ ]:
def activation_velocity(direction, strength, start, end):
    direction = direction / direction.norm()

    def velocity(t, x):
        gate = window_gate(t, start, end)[:, None]
        with torch.no_grad():
            hidden = model.features(t, x)
            feature_scale = hidden.square().mean(dim=1, keepdim=True).sqrt()
            edited = hidden + strength * gate * feature_scale * direction[None, :]
            return model.velocity_from_features(edited)

    return velocity


In [ ]:
set_seed(SEED + 50)
x0 = sample_base(1400, device=device)
target_reference = sample_class(1400, TARGET_CLASS, device=device)

ACTIVATION_STRENGTH = 7.0
ACTIVATION_WINDOW = (0.40, 0.90)
COMBINED_ACTIVATION_STRENGTH = 5.0
COMBINED_ACTIVATION_WINDOW = (0.35, 0.85)


In [ ]:
methods = {
    "baseline": base_velocity(model),
    "activation 0.0": activation_velocity(reference_direction, 0.0, *ACTIVATION_WINDOW),
    "shuffled activation": activation_velocity(
        shuffled_direction, ACTIVATION_STRENGTH, *ACTIVATION_WINDOW,
    ),
    "activation steering": activation_velocity(
        reference_direction, ACTIVATION_STRENGTH, *ACTIVATION_WINDOW,
    ),
    "noise alignment": make_noise_aligned_velocity(
        model, stats, target_class=TARGET_CLASS, strength=1.6
    ),
    "gradient guidance": make_gradient_guided_velocity(
        model, stats, target_class=TARGET_CLASS, strength=3.0, start=0.15, end=0.92
    ),
    "noise + activation": add_noise_alignment(
        activation_velocity(
            reference_direction,
            COMBINED_ACTIVATION_STRENGTH,
            *COMBINED_ACTIVATION_WINDOW,
        ),
        stats,
        target_class=TARGET_CLASS,
        strength=1.6,
    ),
}


In [ ]:
rows, paths = compare_steering_methods(
    methods, x0, target_reference, stats, target_class=TARGET_CLASS, steps=56,
)
comparison_table = pd.DataFrame(rows)
comparison_table["runtime / baseline"] = comparison_table["seconds"] / comparison_table.loc[0, "seconds"]
display(comparison_table.round(4))


In [ ]:
plot_trajectory_comparison(
    {
        "baseline": paths["baseline"],
        "shuffled direction": paths["shuffled activation"],
        "activation steering": paths["activation steering"],
        "noise + activation": paths["noise + activation"],
    },
    target_reference=target_reference[:700],
    n_lines=18,
)
plt.show()
plot_method_tradeoff(comparison_table)


In [ ]:
method_rows = comparison_table.set_index("method")
activation_effects = pd.Series({
    "activation target-rate gain": method_rows.loc["activation steering", "target_rate"] - method_rows.loc["baseline", "target_rate"],
    "shuffled-direction gain": method_rows.loc["shuffled activation", "target_rate"] - method_rows.loc["baseline", "target_rate"],
    "activation Wasserstein reduction": method_rows.loc["baseline", "target_wasserstein"] - method_rows.loc["activation steering", "target_wasserstein"],
})
display(activation_effects.round(4))


## 4. Reading the method comparison

The methods intervene at different places. Noise alignment changes a denoised
estimate early, objective-gradient guidance differentiates a class score, and
activation steering edits a hidden feature. Target-class rate describes
control, Wasserstein distance describes agreement with target examples,
diversity describes spread, and runtime exposes the price of online
backpropagation. No method is best without naming the trade-off that matters.

**Questions for discussion**

1. Does probe accuracy predict steering success at the same collection time?
2. Over which times is the learned direction stable enough to reuse?
3. When does combining coarse high-noise and later activation control help?
4. Which conclusion from 2D would require a new experiment before claiming it for images?

The probe and intervention answer different questions: held-out decoding shows
that class information is readable from the hidden layer, while paired
sampling tests whether one fixed late-time direction is useful over the chosen
intervention window. Neither result implies a unique class axis or a
time-invariant representation. Across the toy series, we have now compared
three distinct intervention sites. Notebook `06` carries only noise alignment
to a pretrained image model and asks whether paired images and batch-level
features still reveal a measurable, partial shift.
